# Analyse des Interactions entre Analogues de Chaînes Latérales dans une Membrane POPC

## Objectif
Ce notebook implémente une analyse quantitative des interactions entre analogues de chaînes latérales d'acides aminés dans une membrane lipidique de POPC. L'objectif est de caractériser la distribution des analogues au sein de la membrane et de contrôler l'effet potentiel des interactions inter-analogues sur les calculs d'énergie libre d'insertion.

## Méthodologie
Deux définitions du contact sont implémentées et comparées :
1. **Définition par distance atomique minimale** : Contact défini par $d_{\min} = \min_{i,j} \|\mathbf{r}_i^A - \mathbf{r}_j^B\|$ entre atomes lourds
2. **Définition par centre de masse** : Contact défini par $d_{\text{COM}} = \|\mathbf{R}_{\text{COM}}^A - \mathbf{R}_{\text{COM}}^B\|$ avec cutoffs dépendant de l'analogue

## Types d'interactions analysées
- Interactions de van der Waals (cutoff typique : 4.0 Å)
- Liaisons hydrogène (cutoff typique : 3.5 Å)
- Interactions hydrophobes (cutoff typique : 4.5 Å)
- Interactions π–π (cutoff typique : 5.0 Å)

## Données d'entrée
Fichiers PDB situés dans `supp_files/output_systems/sc*/*.pdb` correspondant à des frames indépendantes de trajectoires MD.

## 1. Import des bibliothèques requises

Importation des bibliothèques standards pour l'analyse moléculaire :
- **MDAnalysis** : Lecture et manipulation de structures moléculaires
- **Biopython** : Analyse structurale et détection de contacts
- **NumPy/SciPy** : Calculs numériques et analyse statistique
- **Matplotlib** : Visualisation des résultats

### ⚠️ Résolution des problèmes de compatibilité

Si vous rencontrez l'erreur `ValueError: numpy.dtype size changed`, cela indique une incompatibilité binaire entre numpy et h5py. 

**Solution recommandée** : Exécutez ces commandes dans un terminal :

```bash
# Option 1 : Réinstaller h5py avec pip
pip uninstall h5py -y
pip install --no-cache-dir h5py

# Option 2 : Si vous utilisez conda
conda install -c conda-forge h5py --force-reinstall

# Option 3 : Mettre à jour numpy et h5py ensemble
pip install --upgrade numpy h5py
```

Après avoir exécuté l'une de ces commandes, redémarrez le kernel du notebook (Kernel → Restart Kernel).

In [2]:
# =============================================================================
# Import des bibliothèques
# =============================================================================
import os
import glob
import json
import warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.spatial import distance
from scipy.ndimage import uniform_filter1d
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns

# MDAnalysis pour l'analyse de trajectoires
import MDAnalysis as mda
from MDAnalysis.analysis import distances as mda_distances

# Biopython pour l'analyse structurale
# from Bio.PDB import PDBParser, NeighborSearch
# from Bio.PDB.Polypeptide import is_aa

# Configuration des warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Configuration de matplotlib pour des figures publication-ready
plt.rcParams.update({
    'figure.figsize': (10, 8),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 11,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'lines.linewidth': 2,
    'axes.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Palette de couleurs cohérente pour les analyses
COLORS = {
    'vdw': '#1f77b4',      # Bleu
    'hbond': '#2ca02c',     # Vert
    'hydrophobic': '#ff7f0e',  # Orange
    'pipi': '#9467bd',      # Violet
    'any': '#d62728',       # Rouge
}

print("✓ Bibliothèques importées avec succès")
print(f"  - MDAnalysis version: {mda.__version__}")
print(f"  - NumPy version: {np.__version__}")
print(f"  - Pandas version: {pd.__version__}")

✓ Bibliothèques importées avec succès
  - MDAnalysis version: 2.10.0
  - NumPy version: 2.3.5
  - Pandas version: 2.3.3


## 2. Configuration et découverte des données

### 2.1 Chemins et paramètres globaux
Configuration des chemins vers les données et des paramètres d'analyse.

In [3]:
# =============================================================================
# Configuration des chemins et paramètres globaux
# =============================================================================

# Chemin racine du projet (relatif au notebook)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]  # Remonte à la racine du projet

# Chemins vers les données
DATA_DIR = PROJECT_ROOT / "supp_files" / "output_systems"
OUTPUT_DIR = NOTEBOOK_DIR / "results"
FIGURES_DIR = NOTEBOOK_DIR / "figures"

# Créer les répertoires de sortie s'ils n'existent pas
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# =============================================================================
# Paramètres d'analyse
# =============================================================================

# Cutoffs par défaut pour les différents types d'interactions (en Ångströms)
DEFAULT_CUTOFFS = {
    'vdw': 4.0,           # van der Waals: somme des rayons VdW + tolérance
    'hbond': 3.5,         # Liaison H: distance donneur-accepteur typique
    'hydrophobic': 4.5,   # Hydrophobe: distance C-C typique
    'pipi': 5.0,          # π-π: distance entre centroïdes aromatiques
}

# Plages de cutoff à explorer pour l'analyse paramétrique
CUTOFF_RANGES = {
    'min_heavy': np.arange(2.5, 8.1, 0.25),   # Distance min atomes lourds
    'com': np.arange(4.0, 15.1, 0.5),          # Distance entre COM
}

# Définition des atomes pour chaque type d'interaction
ATOM_DEFINITIONS = {
    # Atomes donneurs/accepteurs de liaisons H (N, O avec H attachés ou électrons libres)
    'hbond_donors': {'N', 'O'},
    'hbond_acceptors': {'N', 'O', 'S'},
    
    # Atomes hydrophobes (carbones aliphatiques et certains soufres)
    'hydrophobic': {'C'},  # Seuls les C non polaires seront sélectionnés
    
    # Atomes aromatiques typiques des cycles
    'aromatic_atoms': {'CG', 'CD1', 'CD2', 'CE1', 'CE2', 'CZ', 'CH2', 'NE1'},
}

# Correspondance des codes de dossiers vers les noms de résidus
# Format: dossier -> (nom_résidu, nom_acide_aminé, type)
ANALOGUE_INFO = {
    'sca': ('SCA', 'Alanine', 'aliphatic'),
    'scc': ('SCC', 'Cysteine', 'polar'),
    'sccm': ('SCCM', 'Cysteine-charged', 'charged'),
    'scd': ('SCD', 'Aspartate', 'charged'),
    'scdn': ('SCDN', 'Aspartate-neutral', 'polar'),
    'sce': ('SCE', 'Glutamate', 'charged'),
    'scen': ('SCEN', 'Glutamate-neutral', 'polar'),
    'scf': ('SCF', 'Phenylalanine', 'aromatic'),
    'schd': ('SCHD', 'Histidine-delta', 'aromatic'),
    'sche': ('SCHE', 'Histidine-epsilon', 'aromatic'),
    'schp': ('SCHP', 'Histidine-protonated', 'charged'),
    'sci': ('SCI', 'Isoleucine', 'aliphatic'),
    'sck': ('SCK', 'Lysine', 'charged'),
    'sckn': ('SCKN', 'Lysine-neutral', 'polar'),
    'scl': ('SCL', 'Leucine', 'aliphatic'),
    'scm': ('SCM', 'Methionine', 'aliphatic'),
    'scn': ('SCN', 'Asparagine', 'polar'),
    'scp': ('SCP', 'Proline', 'aliphatic'),
    'scq': ('SCQ', 'Glutamine', 'polar'),
    'scr': ('SCR', 'Arginine', 'charged'),
    'scrn': ('SCRN', 'Arginine-neutral', 'polar'),
    'scs': ('SCS', 'Serine', 'polar'),
    'sct': ('SCT', 'Threonine', 'polar'),
    'scv': ('SCV', 'Valine', 'aliphatic'),
    'scw': ('SCW', 'Tryptophan', 'aromatic'),
    'scy': ('SCY', 'Tyrosine', 'aromatic'),
    'scym': ('SCYM', 'Tyrosine-methyl', 'aromatic'),
}

# Rayon de giration approximatif pour les cutoffs COM dépendant de l'analogue (Å)
# Valeurs approximatives basées sur la taille des chaînes latérales
ANALOGUE_RADII = {
    'sca': 2.0,   # Alanine - petite
    'scc': 2.5,   # Cysteine
    'sccm': 3.0,  # Cysteine-charged
    'scd': 3.5,   # Aspartate
    'scdn': 3.5,
    'sce': 4.0,   # Glutamate
    'scen': 4.0,
    'scf': 4.5,   # Phenylalanine - cycle aromatique
    'schd': 4.5,  # Histidine
    'sche': 4.5,
    'schp': 4.5,
    'sci': 3.5,   # Isoleucine
    'sck': 5.0,   # Lysine - longue chaîne
    'sckn': 5.0,
    'scl': 3.5,   # Leucine
    'scm': 4.0,   # Methionine
    'scn': 3.5,   # Asparagine
    'scp': 3.0,   # Proline
    'scq': 4.0,   # Glutamine
    'scr': 5.5,   # Arginine - très longue
    'scrn': 5.5,
    'scs': 2.5,   # Serine
    'sct': 3.0,   # Threonine
    'scv': 3.0,   # Valine
    'scw': 5.0,   # Tryptophan - grand cycle
    'scy': 4.5,   # Tyrosine
    'scym': 4.5,
}

print(f"✓ Configuration chargée")
print(f"  - Répertoire des données: {DATA_DIR}")
print(f"  - Répertoire de sortie: {OUTPUT_DIR}")
print(f"  - {len(ANALOGUE_INFO)} types d'analogues définis")

✓ Configuration chargée
  - Répertoire des données: /Users/valentin/Dropbox/SC_article/Bories_and_Lague_2025/supp_files/output_systems
  - Répertoire de sortie: /Users/valentin/Dropbox/SC_article/Bories_and_Lague_2025/figures/data/mono_data/cutoff_research/results
  - 27 types d'analogues définis


### 2.2 Découverte automatique des fichiers PDB

Exploration du répertoire `output_systems` pour identifier tous les systèmes disponibles et leurs fichiers PDB associés.

In [4]:
# =============================================================================
# Découverte des fichiers PDB disponibles
# =============================================================================

def discover_pdb_files(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Découvre tous les fichiers PDB dans les sous-répertoires sc* et sc*/traj*.
    
    Parameters
    ----------
    data_dir : Path
        Répertoire racine contenant les sous-dossiers sc*
        
    Returns
    -------
    Dict[str, List[Path]]
        Dictionnaire {nom_système: [liste_fichiers_pdb]}
    """
    systems = {}
    
    # Recherche des dossiers sc*
    sc_dirs = sorted(data_dir.glob("sc*"))
    
    for sc_dir in sc_dirs:
        if not sc_dir.is_dir():
            continue
            
        system_name = sc_dir.name
        pdb_files = []
        
        # Recherche des fichiers PDB directement dans sc_dir
        pdb_files.extend(sorted(sc_dir.glob("*.pdb")))
        
        # Recherche des fichiers PDB dans les sous-dossiers traj*
        traj_dirs = sorted(sc_dir.glob("traj*"))
        for traj_dir in traj_dirs:
            if traj_dir.is_dir():
                pdb_files.extend(sorted(traj_dir.glob("*.pdb")))
        
        # Filtrer les fichiers non vides
        valid_files = [f for f in pdb_files if f.stat().st_size > 1000]
        
        if valid_files:
            systems[system_name] = valid_files
            
    return systems


def get_system_info(systems: Dict[str, List[Path]]) -> pd.DataFrame:
    """
    Génère un tableau récapitulatif des systèmes découverts.
    
    Parameters
    ----------
    systems : Dict[str, List[Path]]
        Dictionnaire des systèmes et fichiers
        
    Returns
    -------
    pd.DataFrame
        Tableau avec informations sur chaque système
    """
    data = []
    for system_name, files in systems.items():
        info = ANALOGUE_INFO.get(system_name, (system_name.upper(), 'Unknown', 'unknown'))
        radius = ANALOGUE_RADII.get(system_name, 3.5)
        
        data.append({
            'Système': system_name,
            'Résidu': info[0],
            'Acide aminé': info[1],
            'Type': info[2],
            'Nb fichiers': len(files),
            'Rayon (Å)': radius,
        })
    
    return pd.DataFrame(data)


# Découverte des systèmes
print("Découverte des fichiers PDB...")
pdb_systems = discover_pdb_files(DATA_DIR)

# Affichage du résumé
systems_df = get_system_info(pdb_systems)
print(f"\n✓ {len(pdb_systems)} systèmes découverts avec {sum(len(f) for f in pdb_systems.values())} fichiers PDB au total\n")

# Affichage du tableau
display(systems_df.style.set_caption("Systèmes d'analogues disponibles"))

Découverte des fichiers PDB...

✓ 27 systèmes découverts avec 46895 fichiers PDB au total



,Système,Résidu,Acide aminé,Type,Nb fichiers,Rayon (Å)
0,sca,SCA,Alanine,aliphatic,1803,2.000000
1,scc,SCC,Cysteine,polar,1803,2.500000
2,sccm,SCCM,Cysteine-charged,charged,1803,3.000000
3,scd,SCD,Aspartate,charged,1802,3.500000
4,scdn,SCDN,Aspartate-neutral,polar,1803,3.500000
5,sce,SCE,Glutamate,charged,1803,4.000000
6,scen,SCEN,Glutamate-neutral,polar,1803,4.000000
7,scf,SCF,Phenylalanine,aromatic,1803,4.500000
8,schd,SCHD,Histidine-delta,aromatic,1803,4.500000
9,sche,SCHE,Histidine-epsilon,aromatic,1640,4.500000


## 3. Structures de données moléculaires

### 3.1 Classe Analogue
Définition d'une classe pour représenter un analogue de chaîne latérale avec ses propriétés atomiques et méthodes de calcul géométrique.

In [5]:
# =============================================================================
# Structures de données pour les analogues
# =============================================================================

@dataclass
class Atom:
    """Représentation d'un atome."""
    name: str
    element: str
    coords: np.ndarray
    index: int
    mass: float = 1.0
    
    def distance_to(self, other: 'Atom') -> float:
        """Calcule la distance euclidienne vers un autre atome."""
        return np.linalg.norm(self.coords - other.coords)


@dataclass 
class Analogue:
    """
    Représentation d'un analogue de chaîne latérale.
    
    Attributes
    ----------
    resid : int
        Numéro de résidu
    resname : str
        Nom du résidu (ex: SCY, SCF)
    atoms : List[Atom]
        Liste de tous les atomes
    """
    resid: int
    resname: str
    atoms: List[Atom] = field(default_factory=list)
    
    # Propriétés cachées calculées à la demande
    _heavy_atoms: Optional[List[Atom]] = field(default=None, repr=False)
    _aromatic_atoms: Optional[List[Atom]] = field(default=None, repr=False)
    _com: Optional[np.ndarray] = field(default=None, repr=False)
    _aromatic_com: Optional[np.ndarray] = field(default=None, repr=False)
    
    @property
    def heavy_atoms(self) -> List[Atom]:
        """Retourne uniquement les atomes lourds (non-H)."""
        if self._heavy_atoms is None:
            self._heavy_atoms = [a for a in self.atoms if a.element != 'H']
        return self._heavy_atoms
    
    @property
    def heavy_coords(self) -> np.ndarray:
        """Coordonnées des atomes lourds sous forme de tableau numpy."""
        return np.array([a.coords for a in self.heavy_atoms])
    
    @property
    def all_coords(self) -> np.ndarray:
        """Coordonnées de tous les atomes."""
        return np.array([a.coords for a in self.atoms])
    
    @property
    def center_of_mass(self) -> np.ndarray:
        """
        Calcule le centre de masse de l'analogue.
        Utilise les masses atomiques standards.
        """
        if self._com is None:
            masses = np.array([ATOMIC_MASSES.get(a.element, 12.0) for a in self.heavy_atoms])
            coords = self.heavy_coords
            self._com = np.average(coords, axis=0, weights=masses)
        return self._com
    
    @property
    def aromatic_atoms(self) -> List[Atom]:
        """Retourne les atomes du cycle aromatique (si présent)."""
        if self._aromatic_atoms is None:
            aromatic_names = ATOM_DEFINITIONS['aromatic_atoms']
            self._aromatic_atoms = [a for a in self.atoms if a.name in aromatic_names]
        return self._aromatic_atoms
    
    @property
    def aromatic_centroid(self) -> Optional[np.ndarray]:
        """
        Calcule le centroïde du cycle aromatique.
        Retourne None si l'analogue n'a pas de cycle aromatique.
        """
        if self._aromatic_com is None:
            arom = self.aromatic_atoms
            if len(arom) >= 5:  # Minimum pour un cycle aromatique
                self._aromatic_com = np.mean([a.coords for a in arom], axis=0)
        return self._aromatic_com
    
    @property
    def has_aromatic_ring(self) -> bool:
        """Vérifie si l'analogue possède un cycle aromatique."""
        return len(self.aromatic_atoms) >= 5
    
    def get_donors(self) -> List[Atom]:
        """Retourne les atomes donneurs de liaison H (N, O avec H)."""
        return [a for a in self.heavy_atoms if a.element in {'N', 'O'}]
    
    def get_acceptors(self) -> List[Atom]:
        """Retourne les atomes accepteurs de liaison H."""
        return [a for a in self.heavy_atoms if a.element in {'N', 'O', 'S'}]
    
    def get_hydrophobic_atoms(self) -> List[Atom]:
        """
        Retourne les atomes hydrophobes (carbones non polaires).
        Exclut les carbones du backbone et carbonyles.
        """
        polar_carbons = {'C', 'CA', 'C1', 'C2', 'C3'}  # Backbone/carbonyles typiques
        return [a for a in self.atoms 
                if a.element == 'C' and a.name not in polar_carbons]


# Masses atomiques standards (en u.m.a.)
ATOMIC_MASSES = {
    'H': 1.008,
    'C': 12.011,
    'N': 14.007,
    'O': 15.999,
    'S': 32.065,
    'P': 30.974,
}


@dataclass
class Frame:
    """
    Représentation d'une frame (snapshot) du système.
    
    Attributes
    ----------
    file_path : Path
        Chemin vers le fichier PDB
    system_name : str
        Nom du système (ex: scy, scf)
    analogues : List[Analogue]
        Liste des analogues dans cette frame
    box : Optional[np.ndarray]
        Dimensions de la boîte de simulation
    """
    file_path: Path
    system_name: str
    analogues: List[Analogue] = field(default_factory=list)
    box: Optional[np.ndarray] = None
    
    @property
    def n_analogues(self) -> int:
        """Nombre d'analogues dans la frame."""
        return len(self.analogues)
    
    def get_analogue_by_resid(self, resid: int) -> Optional[Analogue]:
        """Trouve un analogue par son numéro de résidu."""
        for ana in self.analogues:
            if ana.resid == resid:
                return ana
        return None


print("✓ Structures de données définies")

✓ Structures de données définies


### 3.2 Chargement des structures PDB

Fonctions pour lire les fichiers PDB et extraire les analogues avec leurs propriétés atomiques.

In [6]:
# =============================================================================
# Chargement des fichiers PDB avec MDAnalysis (optimisé avec cache et ThreadPool)
# =============================================================================

import pickle
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

# Fichier de cache pour éviter de recharger les données
CACHE_FILE = DATA_DIR / "frames_cache.pkl"


def load_frame_mda(pdb_path: Path, system_name: str) -> Optional[Frame]:
    """
    Charge une frame PDB et extrait les analogues avec MDAnalysis.
    Retourne None en cas d'erreur.
    """
    try:
        # Déterminer le nom de résidu attendu
        resname = ANALOGUE_INFO.get(system_name, (system_name.upper(),))[0]
        
        # Charger la structure avec MDAnalysis
        u = mda.Universe(str(pdb_path))
        
        # Extraire les dimensions de la boîte (si disponibles)
        box = u.dimensions[:3] if u.dimensions is not None else None
        
        # Sélectionner les analogues
        try:
            selection = u.select_atoms(f"resname {resname}")
        except:
            selection = u.select_atoms("not (resname POPC* or resname TIP* or resname SOD or resname CLA or resname POT)")
        
        if len(selection) == 0:
            return None
        
        # Grouper par résidu
        analogues = []
        for res in selection.residues:
            atoms_list = []
            for atom in res.atoms:
                element = atom.element if hasattr(atom, 'element') and atom.element else atom.name[0]
                atoms_list.append(Atom(
                    name=atom.name,
                    element=element,
                    coords=atom.position.copy(),
                    index=atom.index,
                    mass=ATOMIC_MASSES.get(element, 12.0)
                ))
            analogues.append(Analogue(resid=res.resid, resname=res.resname, atoms=atoms_list))
        
        return Frame(file_path=pdb_path, system_name=system_name, analogues=analogues, box=box)
    except Exception:
        return None


def load_system_frames_threaded(system_name: str, 
                                pdb_files: List[Path],
                                max_frames: Optional[int] = None,
                                n_workers: int = 4,
                                show_progress: bool = False) -> List[Frame]:
    """
    Charge les frames pour un système avec des threads.
    """
    files_to_load = pdb_files[:max_frames] if max_frames else pdb_files
    
    def load_single(pdb_path):
        return load_frame_mda(pdb_path, system_name)
    
    frames = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        if show_progress:
            results = list(tqdm(
                executor.map(load_single, files_to_load),
                total=len(files_to_load),
                desc=f"  {system_name}",
                leave=False
            ))
        else:
            results = list(executor.map(load_single, files_to_load))
        
        for frame in results:
            if frame is not None and frame.n_analogues > 0:
                frames.append(frame)
    
    return frames


def load_all_frames_with_cache(systems: Dict[str, List[Path]], 
                               max_frames_per_system: Optional[int] = None,
                               use_cache: bool = True,
                               n_workers: int = 4,
                               verbose: bool = True) -> Dict[str, List[Frame]]:
    """
    Charge toutes les frames avec mise en cache et threads.
    
    Parameters
    ----------
    systems : Dict[str, List[Path]]
        Dictionnaire {nom_système: [chemins_pdb]}
    max_frames_per_system : int, optional
        Limite le nombre de frames par système
    use_cache : bool
        Utiliser le cache si disponible (défaut: True)
    n_workers : int
        Nombre de threads pour le chargement parallèle
    verbose : bool
        Affiche la progression
        
    Returns
    -------
    Dict[str, List[Frame]]
        Dictionnaire {nom_système: [frames]}
    """
    # Essayer de charger depuis le cache
    if use_cache and CACHE_FILE.exists():
        if verbose:
            print(f"📦 Chargement depuis le cache: {CACHE_FILE.name}")
        try:
            with open(CACHE_FILE, 'rb') as f:
                cached_data = pickle.load(f)
            # Vérifier que le cache correspond aux systèmes demandés
            if set(cached_data.keys()) == set(systems.keys()):
                if verbose:
                    total_frames = sum(len(frames) for frames in cached_data.values())
                    print(f"✓ Cache valide: {len(cached_data)} systèmes, {total_frames} frames")
                return cached_data
            else:
                if verbose:
                    print("⚠ Cache invalide (systèmes différents), rechargement...")
        except Exception as e:
            if verbose:
                print(f"⚠ Erreur de lecture du cache: {e}")
    
    # Chargement depuis les fichiers
    all_frames = {}
    total_systems = len(systems)
    
    for idx, (system_name, pdb_files) in enumerate(systems.items(), 1):
        n_files = min(len(pdb_files), max_frames_per_system) if max_frames_per_system else len(pdb_files)
        if verbose:
            print(f"  [{idx}/{total_systems}] {system_name} ({n_files} fichiers)...", end=" ", flush=True)
        
        # Chargement avec threads
        frames = load_system_frames_threaded(
            system_name, pdb_files, max_frames_per_system, n_workers, show_progress=False
        )
        
        if frames:
            all_frames[system_name] = frames
            if verbose:
                print(f"✓ {len(frames)} frames")
        else:
            if verbose:
                print("✗ Aucune frame")
    
    # Sauvegarder dans le cache
    if use_cache and all_frames:
        if verbose:
            print(f"\n💾 Sauvegarde du cache: {CACHE_FILE.name}")
        try:
            with open(CACHE_FILE, 'wb') as f:
                pickle.dump(all_frames, f, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as e:
            if verbose:
                print(f"⚠ Erreur de sauvegarde du cache: {e}")
    
    return all_frames


def clear_cache():
    """Supprime le fichier de cache."""
    if CACHE_FILE.exists():
        CACHE_FILE.unlink()
        print(f"✓ Cache supprimé: {CACHE_FILE.name}")
    else:
        print("Pas de cache à supprimer")


# =============================================================================
# Chargement des données
# =============================================================================

# Pour forcer le rechargement depuis les fichiers PDB, décommentez:
# clear_cache()

print("Chargement des structures PDB...")
print(f"  (Cache: {CACHE_FILE.name})")
print()

# Nombre de threads pour le chargement parallèle
N_WORKERS = 8

all_frames = load_all_frames_with_cache(
    pdb_systems,
    max_frames_per_system=100,  # Mettre un nombre pour limiter (ex: 100)
    use_cache=True,
    n_workers=N_WORKERS
)

print(f"\n✓ Chargement terminé: {len(all_frames)} systèmes valides")
print(f"  Total de frames: {sum(len(frames) for frames in all_frames.values())}")

Chargement des structures PDB...
  (Cache: frames_cache.pkl)

📦 Chargement depuis le cache: frames_cache.pkl
✓ Cache valide: 27 systèmes, 2700 frames

✓ Chargement terminé: 27 systèmes valides
  Total de frames: 2700


## 4. Définitions des contacts et calculs de distances

### 4.1 Définition 1 : Distance minimale entre atomes lourds

La première définition du contact est basée sur la distance minimale entre paires d'atomes lourds appartenant à deux analogues distincts :

$$d_{\min}(A, B) = \min_{i \in A, j \in B} \|\mathbf{r}_i - \mathbf{r}_j\|$$

où $\mathbf{r}_i$ et $\mathbf{r}_j$ sont les positions des atomes lourds des analogues A et B.

In [7]:
# =============================================================================
# Calcul de distances entre analogues
# =============================================================================

def min_heavy_atom_distance(ana1: Analogue, ana2: Analogue) -> float:
    """
    Calcule la distance minimale entre atomes lourds de deux analogues.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float
        Distance minimale en Ångströms
    """
    coords1 = ana1.heavy_coords
    coords2 = ana2.heavy_coords
    
    # Calcul efficace avec scipy
    dist_matrix = distance.cdist(coords1, coords2)
    return dist_matrix.min()


def com_distance(ana1: Analogue, ana2: Analogue) -> float:
    """
    Calcule la distance entre les centres de masse de deux analogues.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float
        Distance COM-COM en Ångströms
    """
    return np.linalg.norm(ana1.center_of_mass - ana2.center_of_mass)


def aromatic_centroid_distance(ana1: Analogue, ana2: Analogue) -> Optional[float]:
    """
    Calcule la distance entre les centroïdes des cycles aromatiques.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
        
    Returns
    -------
    float or None
        Distance entre centroïdes, ou None si un des analogues n'a pas de cycle
    """
    c1 = ana1.aromatic_centroid
    c2 = ana2.aromatic_centroid
    
    if c1 is None or c2 is None:
        return None
    
    return np.linalg.norm(c1 - c2)


def compute_pairwise_distances(frame: Frame) -> Dict[str, np.ndarray]:
    """
    Calcule toutes les distances par paires d'analogues pour une frame.
    
    Parameters
    ----------
    frame : Frame
        La frame à analyser
        
    Returns
    -------
    Dict[str, np.ndarray]
        Dictionnaire contenant:
        - 'min_heavy': matrice des distances minimales atomes lourds
        - 'com': matrice des distances COM-COM
        - 'aromatic': matrice des distances centroïde-centroïde (NaN si non aromatique)
        - 'pairs': liste des paires (i, j) avec i < j
    """
    n = frame.n_analogues
    analogues = frame.analogues
    
    # Initialisation des matrices
    min_heavy = np.zeros((n, n))
    com_dist = np.zeros((n, n))
    aromatic_dist = np.full((n, n), np.nan)
    
    pairs = []
    
    for i in range(n):
        for j in range(i + 1, n):
            ana1, ana2 = analogues[i], analogues[j]
            
            # Distance minimale atomes lourds
            d_min = min_heavy_atom_distance(ana1, ana2)
            min_heavy[i, j] = min_heavy[j, i] = d_min
            
            # Distance COM
            d_com = com_distance(ana1, ana2)
            com_dist[i, j] = com_dist[j, i] = d_com
            
            # Distance aromatique (si applicable)
            d_arom = aromatic_centroid_distance(ana1, ana2)
            if d_arom is not None:
                aromatic_dist[i, j] = aromatic_dist[j, i] = d_arom
            
            pairs.append((i, j))
    
    return {
        'min_heavy': min_heavy,
        'com': com_dist,
        'aromatic': aromatic_dist,
        'pairs': pairs,
    }


# Test sur une frame
if all_frames:
    test_system = list(all_frames.keys())[0]
    test_frame = all_frames[test_system][0]
    
    print(f"Test de calcul des distances sur {test_system}...")
    distances = compute_pairwise_distances(test_frame)
    
    # Statistiques
    min_heavy_flat = distances['min_heavy'][np.triu_indices(test_frame.n_analogues, k=1)]
    com_flat = distances['com'][np.triu_indices(test_frame.n_analogues, k=1)]
    
    print(f"\n  Distance min atomes lourds:")
    print(f"    Min: {min_heavy_flat.min():.2f} Å, Max: {min_heavy_flat.max():.2f} Å, Moyenne: {min_heavy_flat.mean():.2f} Å")
    
    print(f"\n  Distance COM-COM:")
    print(f"    Min: {com_flat.min():.2f} Å, Max: {com_flat.max():.2f} Å, Moyenne: {com_flat.mean():.2f} Å")
    
    print("\n✓ Fonctions de calcul de distances opérationnelles")

Test de calcul des distances sur sca...

  Distance min atomes lourds:
    Min: 3.55 Å, Max: 84.22 Å, Moyenne: 34.10 Å

  Distance COM-COM:
    Min: 3.55 Å, Max: 84.22 Å, Moyenne: 34.10 Å

✓ Fonctions de calcul de distances opérationnelles


### 4.2 Définition 2 : Distance entre centres de masse avec cutoffs dépendant de l'analogue

La seconde définition utilise la distance entre centres de masse avec un cutoff adapté à la taille des analogues :

$$d_{\text{COM}}(A, B) = \|\mathbf{R}_{\text{COM}}^A - \mathbf{R}_{\text{COM}}^B\|$$

Le contact est établi si $d_{\text{COM}} < r_A + r_B + d_{\text{tol}}$, où $r_A$ et $r_B$ sont les rayons effectifs des analogues.

In [8]:
# =============================================================================
# Fonctions de détection de contacts avec cutoff adaptatif
# =============================================================================

def compute_adaptive_cutoff(system_name: str, tolerance: float = 2.0) -> float:
    """
    Calcule un cutoff COM adapté au système.
    
    Le cutoff est basé sur la somme des rayons effectifs (2 * rayon) 
    plus une tolérance pour les contacts.
    
    Parameters
    ----------
    system_name : str
        Nom du système
    tolerance : float
        Tolérance additionnelle en Å
        
    Returns
    -------
    float
        Cutoff adaptatif en Å
    """
    radius = ANALOGUE_RADII.get(system_name, 3.5)
    return 2 * radius + tolerance


def detect_contacts_min_heavy(frame: Frame, cutoff: float) -> List[Tuple[int, int, float]]:
    """
    Détecte les contacts basés sur la distance minimale atomes lourds.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance en Å
        
    Returns
    -------
    List[Tuple[int, int, float]]
        Liste de (resid1, resid2, distance) pour les contacts
    """
    contacts = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = min_heavy_atom_distance(analogues[i], analogues[j])
            if dist <= cutoff:
                contacts.append((analogues[i].resid, analogues[j].resid, dist))
    
    return contacts


def detect_contacts_com(frame: Frame, cutoff: float) -> List[Tuple[int, int, float]]:
    """
    Détecte les contacts basés sur la distance COM-COM.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance en Å
        
    Returns
    -------
    List[Tuple[int, int, float]]
        Liste de (resid1, resid2, distance) pour les contacts
    """
    contacts = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = com_distance(analogues[i], analogues[j])
            if dist <= cutoff:
                contacts.append((analogues[i].resid, analogues[j].resid, dist))
    
    return contacts


def count_contacts_per_analogue(contacts: List[Tuple[int, int, float]], 
                                 n_analogues: int) -> np.ndarray:
    """
    Compte le nombre de contacts par analogue.
    
    Parameters
    ----------
    contacts : List[Tuple[int, int, float]]
        Liste des contacts détectés
    n_analogues : int
        Nombre total d'analogues
        
    Returns
    -------
    np.ndarray
        Nombre de contacts pour chaque analogue
    """
    counts = np.zeros(n_analogues, dtype=int)
    for resid1, resid2, _ in contacts:
        counts[resid1 - 1] += 1
        counts[resid2 - 1] += 1
    return counts


print("✓ Fonctions de détection de contacts définies")

✓ Fonctions de détection de contacts définies


## 5. Détection par types d'interactions

### 5.1 Interactions de van der Waals

Les interactions de van der Waals sont détectées sur la base de la distance entre atomes lourds. Un contact VdW est établi lorsque la distance inter-atomique est inférieure à la somme des rayons de van der Waals plus une tolérance :

$$d_{ij} \leq r_i^{\text{VdW}} + r_j^{\text{VdW}} + d_{\text{tol}}$$

Le cutoff typique est de 4.0 Å pour une détection générale.

In [9]:
# =============================================================================
# Détection des interactions de van der Waals
# =============================================================================

# Rayons de van der Waals standards (en Å)
#Rayons de van der Waals extraits de la série de volumes de Bondi (1964)
VDW_RADII = {
    'H': 1.20,
    'C': 1.70,
    'N': 1.55,
    'O': 1.52,
    'S': 1.80,
    'P': 1.80,
}


def detect_vdw_contacts(ana1: Analogue, ana2: Analogue, 
                        cutoff: float = 4.0) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les contacts de van der Waals entre deux analogues.
    
    Un contact VdW est établi si au moins une paire d'atomes lourds
    est à une distance inférieure au cutoff.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance en Å (défaut: 4.0)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_en_contact)
    """
    coords1 = ana1.heavy_coords
    coords2 = ana2.heavy_coords
    
    # Calcul de la matrice de distances
    dist_matrix = distance.cdist(coords1, coords2)
    
    # Distance minimale
    min_dist = dist_matrix.min()
    
    # Trouver toutes les paires en contact
    contact_pairs = []
    if min_dist <= cutoff:
        indices = np.where(dist_matrix <= cutoff)
        for idx1, idx2 in zip(indices[0], indices[1]):
            atom1 = ana1.heavy_atoms[idx1]
            atom2 = ana2.heavy_atoms[idx2]
            contact_pairs.append((atom1.name, atom2.name, dist_matrix[idx1, idx2]))
    
    return (min_dist <= cutoff, min_dist, contact_pairs)


def analyze_vdw_frame(frame: Frame, cutoff: float = 4.0) -> Dict:
    """
    Analyse complète des contacts VdW pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance VdW
        
    Returns
    -------
    Dict
        Résultats de l'analyse incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_vdw_contacts(
                analogues[i], analogues[j], cutoff
            )
            all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_atom_pairs': len(pairs),
                    'atom_pairs': pairs[:5],  # Garder les 5 premières paires
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'n_pairs_total': len(all_distances),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection VdW définie")

✓ Détection VdW définie


### 5.2 Liaisons hydrogène

Les liaisons hydrogène sont détectées entre atomes donneurs (N, O avec H attaché) et accepteurs (N, O, S avec paires d'électrons libres). Le critère géométrique standard est :

$$d_{\text{D-A}} \leq 3.5 \text{ Å}$$

où D est l'atome donneur et A l'accepteur. Pour une analyse plus rigoureuse, on peut aussi vérifier l'angle D-H...A.

In [10]:
# =============================================================================
# Détection des liaisons hydrogène
# =============================================================================

def detect_hbond_contacts(ana1: Analogue, ana2: Analogue, 
                          cutoff: float = 3.5) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les liaisons hydrogène potentielles entre deux analogues.
    
    Basé sur la distance donneur-accepteur (D-A). Les atomes donneurs sont
    N et O pouvant porter un H. Les accepteurs sont N, O et S.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance D-A en Å (défaut: 3.5)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_H-bond)
    """
    donors1 = ana1.get_donors()
    acceptors1 = ana1.get_acceptors()
    donors2 = ana2.get_donors()
    acceptors2 = ana2.get_acceptors()
    
    if not (donors1 or donors2) or not (acceptors1 or acceptors2):
        return (False, np.inf, [])
    
    hbond_pairs = []
    min_dist = np.inf
    
    # Ana1 donneur -> Ana2 accepteur
    for d in donors1:
        for a in acceptors2:
            dist = d.distance_to(a)
            min_dist = min(min_dist, dist)
            if dist <= cutoff:
                hbond_pairs.append((d.name, a.name, dist))
    
    # Ana2 donneur -> Ana1 accepteur
    for d in donors2:
        for a in acceptors1:
            dist = d.distance_to(a)
            min_dist = min(min_dist, dist)
            if dist <= cutoff:
                hbond_pairs.append((d.name, a.name, dist))
    
    return (len(hbond_pairs) > 0, min_dist, hbond_pairs)


def analyze_hbond_frame(frame: Frame, cutoff: float = 3.5) -> Dict:
    """
    Analyse complète des liaisons H potentielles pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance D-A
        
    Returns
    -------
    Dict
        Résultats incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_hbond_contacts(
                analogues[i], analogues[j], cutoff
            )
            
            if min_dist < np.inf:
                all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_hbonds': len(pairs),
                    'hbond_pairs': pairs,
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection liaisons H définie")

✓ Détection liaisons H définie


### 5.3 Interactions hydrophobes

Les interactions hydrophobes sont détectées entre atomes de carbone aliphatiques (non polaires). Le critère est basé sur la distance C-C :

$$d_{\text{C-C}} \leq 4.5 \text{ Å}$$

Seuls les carbones non liés à des hétéroatomes (N, O, S) sont considérés comme hydrophobes.

In [11]:
# =============================================================================
# Détection des interactions hydrophobes
# =============================================================================

def detect_hydrophobic_contacts(ana1: Analogue, ana2: Analogue, 
                                 cutoff: float = 4.5) -> Tuple[bool, float, List[Tuple[str, str, float]]]:
    """
    Détecte les contacts hydrophobes entre deux analogues.
    
    Basé sur les distances entre atomes de carbone (considérés hydrophobes).
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance C-C en Å (défaut: 4.5)
        
    Returns
    -------
    Tuple[bool, float, List[Tuple[str, str, float]]]
        (contact_detecté, distance_minimale, liste_paires_hydrophobes)
    """
    hydro1 = ana1.get_hydrophobic_atoms()
    hydro2 = ana2.get_hydrophobic_atoms()
    
    if not hydro1 or not hydro2:
        return (False, np.inf, [])
    
    # Coordonnées des atomes hydrophobes
    coords1 = np.array([a.coords for a in hydro1])
    coords2 = np.array([a.coords for a in hydro2])
    
    # Matrice de distances
    dist_matrix = distance.cdist(coords1, coords2)
    min_dist = dist_matrix.min()
    
    # Paires en contact
    hydro_pairs = []
    if min_dist <= cutoff:
        indices = np.where(dist_matrix <= cutoff)
        for idx1, idx2 in zip(indices[0], indices[1]):
            hydro_pairs.append((
                hydro1[idx1].name, 
                hydro2[idx2].name, 
                dist_matrix[idx1, idx2]
            ))
    
    return (len(hydro_pairs) > 0, min_dist, hydro_pairs)


def analyze_hydrophobic_frame(frame: Frame, cutoff: float = 4.5) -> Dict:
    """
    Analyse complète des contacts hydrophobes pour une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance C-C
        
    Returns
    -------
    Dict
        Résultats incluant contacts et statistiques
    """
    contacts = []
    all_distances = []
    analogues = frame.analogues
    n = len(analogues)
    
    for i in range(n):
        for j in range(i + 1, n):
            has_contact, min_dist, pairs = detect_hydrophobic_contacts(
                analogues[i], analogues[j], cutoff
            )
            
            if min_dist < np.inf:
                all_distances.append(min_dist)
            
            if has_contact:
                contacts.append({
                    'resid1': analogues[i].resid,
                    'resid2': analogues[j].resid,
                    'min_distance': min_dist,
                    'n_contacts': len(pairs),
                    'contact_pairs': pairs[:5],
                })
    
    return {
        'contacts': contacts,
        'n_contacts': len(contacts),
        'distances': np.array(all_distances),
        'fraction_in_contact': len(contacts) / len(all_distances) if all_distances else 0,
    }


print("✓ Détection contacts hydrophobes définie")

✓ Détection contacts hydrophobes définie


### 5.4 Interactions π–π (stacking aromatique)

Les interactions π–π sont détectées entre cycles aromatiques. Le critère principal est basé sur la distance entre les centroïdes des cycles :

$$d_{\text{centroid}} \leq 5.0 \text{ Å}$$

Cette analyse s'applique uniquement aux analogues aromatiques (Phe, Tyr, Trp, His).

In [12]:
# =============================================================================
# Détection des interactions π–π
# =============================================================================

def compute_ring_normal(atoms: List[Atom]) -> Optional[np.ndarray]:
    """
    Calcule le vecteur normal au plan du cycle aromatique.
    
    Utilise les 3 premiers atomes pour définir le plan par produit vectoriel.
    
    Parameters
    ----------
    atoms : List[Atom]
        Atomes du cycle aromatique (minimum 3)
        
    Returns
    -------
    np.ndarray or None
        Vecteur normal unitaire, ou None si < 3 atomes
    """
    if len(atoms) < 3:
        return None
    
    # Vecteurs dans le plan du cycle
    v1 = atoms[1].coords - atoms[0].coords
    v2 = atoms[2].coords - atoms[0].coords
    
    # Produit vectoriel
    normal = np.cross(v1, v2)
    norm = np.linalg.norm(normal)
    
    if norm < 1e-6:
        return None
    
    return normal / norm


def compute_ring_angle(normal1: np.ndarray, normal2: np.ndarray) -> float:
    """
    Calcule l'angle entre deux plans aromatiques.
    
    Parameters
    ----------
    normal1, normal2 : np.ndarray
        Vecteurs normaux aux plans
        
    Returns
    -------
    float
        Angle en degrés (0° = parallèle, 90° = perpendiculaire)
    """
    cos_angle = abs(np.dot(normal1, normal2))
    cos_angle = min(1.0, max(-1.0, cos_angle))  # Clamp pour éviter erreurs numériques
    return np.degrees(np.arccos(cos_angle))


def detect_pipi_contacts(ana1: Analogue, ana2: Analogue, 
                         cutoff: float = 5.0) -> Tuple[bool, Optional[float], Optional[float]]:
    """
    Détecte les interactions π–π entre deux analogues aromatiques.
    
    Parameters
    ----------
    ana1, ana2 : Analogue
        Les deux analogues à comparer
    cutoff : float
        Seuil de distance centroïde-centroïde en Å (défaut: 5.0)
        
    Returns
    -------
    Tuple[bool, Optional[float], Optional[float]]
        (contact_detecté, distance_centroïdes, angle_entre_plans)
    """
    # Vérifier que les deux analogues ont des cycles aromatiques
    if not ana1.has_aromatic_ring or not ana2.has_aromatic_ring:
        return (False, None, None)
    
    # Distance entre centroïdes
    c1 = ana1.aromatic_centroid
    c2 = ana2.aromatic_centroid
    dist = np.linalg.norm(c1 - c2)
    
    # Angle entre les plans
    normal1 = compute_ring_normal(ana1.aromatic_atoms)
    normal2 = compute_ring_normal(ana2.aromatic_atoms)
    
    angle = None
    if normal1 is not None and normal2 is not None:
        angle = compute_ring_angle(normal1, normal2)
    
    return (dist <= cutoff, dist, angle)


print("✓ Détection π–π définie")

✓ Détection π–π définie


In [13]:
# =============================================================================
# Analyse des analogues isolés en fonction du cutoff
# =============================================================================

from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline
import scipy.stats as stats


@dataclass
class IsolationAnalysisResult:
    """Résultats de l'analyse d'isolation pour un système."""
    system_name: str
    cutoffs: np.ndarray
    fraction_isolated: np.ndarray          # Fraction moyenne d'analogues isolés
    fraction_isolated_std: np.ndarray      # Écart-type
    fraction_isolated_ci_low: np.ndarray   # IC 95% bas
    fraction_isolated_ci_high: np.ndarray  # IC 95% haut
    n_frames: int
    n_analogues_per_frame: float
    
    # Métriques dérivées
    optimal_cutoff_inflection: Optional[float] = None
    optimal_cutoff_elbow: Optional[float] = None
    optimal_cutoff_percolation: Optional[float] = None
    
    
def compute_isolation_fraction(frame: Frame, cutoff: float, 
                                distance_type: str = 'min_heavy') -> Tuple[float, int, int]:
    """
    Calcule la fraction d'analogues isolés (sans contact) dans une frame.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance en Å
    distance_type : str
        'min_heavy' pour distance minimale atomes lourds, 'com' pour centre de masse
        
    Returns
    -------
    Tuple[float, int, int]
        (fraction_isolés, nombre_isolés, nombre_total)
    """
    n = frame.n_analogues
    if n < 2:
        return (1.0, n, n)
    
    analogues = frame.analogues
    has_contact = np.zeros(n, dtype=bool)
    
    # Fonction de distance selon le type
    dist_func = min_heavy_atom_distance if distance_type == 'min_heavy' else com_distance
    
    for i in range(n):
        for j in range(i + 1, n):
            d = dist_func(analogues[i], analogues[j])
            if d <= cutoff:
                has_contact[i] = True
                has_contact[j] = True
    
    n_isolated = np.sum(~has_contact)
    return (n_isolated / n, n_isolated, n)


def compute_contact_degree_distribution(frame: Frame, cutoff: float,
                                         distance_type: str = 'min_heavy') -> np.ndarray:
    """
    Calcule la distribution du nombre de contacts par analogue.
    
    Parameters
    ----------
    frame : Frame
        Frame à analyser
    cutoff : float
        Seuil de distance
    distance_type : str
        Type de distance ('min_heavy' ou 'com')
        
    Returns
    -------
    np.ndarray
        Nombre de contacts pour chaque analogue
    """
    n = frame.n_analogues
    if n < 2:
        return np.zeros(n, dtype=int)
    
    analogues = frame.analogues
    contact_counts = np.zeros(n, dtype=int)
    dist_func = min_heavy_atom_distance if distance_type == 'min_heavy' else com_distance
    
    for i in range(n):
        for j in range(i + 1, n):
            d = dist_func(analogues[i], analogues[j])
            if d <= cutoff:
                contact_counts[i] += 1
                contact_counts[j] += 1
    
    return contact_counts


def analyze_isolation_vs_cutoff(frames: List[Frame], 
                                 cutoffs: np.ndarray,
                                 distance_type: str = 'min_heavy',
                                 n_bootstrap: int = 1000,
                                 confidence_level: float = 0.95) -> IsolationAnalysisResult:
    """
    Analyse complète de la fraction d'analogues isolés en fonction du cutoff.
    
    Parameters
    ----------
    frames : List[Frame]
        Liste des frames à analyser
    cutoffs : np.ndarray
        Plage de cutoffs à explorer (Å)
    distance_type : str
        Type de distance ('min_heavy' ou 'com')
    n_bootstrap : int
        Nombre d'itérations bootstrap pour les intervalles de confiance
    confidence_level : float
        Niveau de confiance pour les IC (défaut: 0.95)
        
    Returns
    -------
    IsolationAnalysisResult
        Résultats complets de l'analyse
    """
    n_cutoffs = len(cutoffs)
    n_frames = len(frames)
    
    # Matrice des fractions isolées : (n_frames, n_cutoffs)
    isolation_matrix = np.zeros((n_frames, n_cutoffs))
    
    for i, frame in enumerate(frames):
        for j, cutoff in enumerate(cutoffs):
            frac, _, _ = compute_isolation_fraction(frame, cutoff, distance_type)
            isolation_matrix[i, j] = frac
    
    # Statistiques descriptives
    mean_isolated = np.mean(isolation_matrix, axis=0)
    std_isolated = np.std(isolation_matrix, axis=0, ddof=1)
    
    # Intervalles de confiance par bootstrap
    alpha = 1 - confidence_level
    ci_low = np.zeros(n_cutoffs)
    ci_high = np.zeros(n_cutoffs)
    
    rng = np.random.default_rng(42)  # Reproductibilité
    for j in range(n_cutoffs):
        data = isolation_matrix[:, j]
        bootstrap_means = np.array([
            np.mean(rng.choice(data, size=n_frames, replace=True))
            for _ in range(n_bootstrap)
        ])
        ci_low[j] = np.percentile(bootstrap_means, 100 * alpha / 2)
        ci_high[j] = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    
    # Nombre moyen d'analogues par frame
    n_analogues_avg = np.mean([f.n_analogues for f in frames])
    
    # Créer le résultat
    system_name = frames[0].system_name if frames else "unknown"
    result = IsolationAnalysisResult(
        system_name=system_name,
        cutoffs=cutoffs,
        fraction_isolated=mean_isolated,
        fraction_isolated_std=std_isolated,
        fraction_isolated_ci_low=ci_low,
        fraction_isolated_ci_high=ci_high,
        n_frames=n_frames,
        n_analogues_per_frame=n_analogues_avg,
    )
    
    # Calcul des cutoffs optimaux
    result.optimal_cutoff_inflection = find_inflection_point(cutoffs, mean_isolated)
    result.optimal_cutoff_elbow = find_elbow_point(cutoffs, mean_isolated)
    result.optimal_cutoff_percolation = find_percolation_threshold(cutoffs, mean_isolated)
    
    return result


def find_inflection_point(x: np.ndarray, y: np.ndarray, 
                          smooth_window: int = 5) -> Optional[float]:
    """
    Trouve le point d'inflexion de la courbe y(x).
    
    Le point d'inflexion correspond au maximum de |d²y/dx²|, indiquant
    la transition la plus rapide dans la fraction d'isolés.
    
    Parameters
    ----------
    x : np.ndarray
        Valeurs de cutoff
    y : np.ndarray
        Fraction d'isolés
    smooth_window : int
        Fenêtre de lissage Savitzky-Golay (doit être impair)
        
    Returns
    -------
    float or None
        Cutoff au point d'inflexion
    """
    if len(x) < smooth_window:
        return None
    
    try:
        # Lissage pour réduire le bruit
        if smooth_window > 2:
            # window_length doit être impair et <= len(y)
            window_len = min(smooth_window, len(y))
            if window_len % 2 == 0:
                window_len -= 1  # Assurer que c'est impair
            if window_len >= 3:  # Minimum requis pour polyorder=2
                y_smooth = savgol_filter(y, window_len, 2)
            else:
                y_smooth = y
        else:
            y_smooth = y
        
        # Dérivée seconde par différences finies
        dy = np.gradient(y_smooth, x)
        d2y = np.gradient(dy, x)
        
        # Point d'inflexion = maximum de |d²y|
        idx = np.argmax(np.abs(d2y))
        return x[idx]
    except Exception:
        return None


def find_elbow_point(x: np.ndarray, y: np.ndarray) -> Optional[float]:
    """
    Trouve le point de coude (elbow) par la méthode de la distance maximale.
    
    Le coude est le point le plus éloigné de la ligne reliant le premier
    et le dernier point de la courbe.
    
    Parameters
    ----------
    x : np.ndarray
        Valeurs de cutoff
    y : np.ndarray
        Fraction d'isolés
        
    Returns
    -------
    float or None
        Cutoff au point de coude
    """
    if len(x) < 3:
        return None
    
    # Normalisation pour traiter x et y de manière équivalente
    x_norm = (x - x.min()) / (x.max() - x.min())
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-10)
    
    # Ligne entre premier et dernier point
    p1 = np.array([x_norm[0], y_norm[0]])
    p2 = np.array([x_norm[-1], y_norm[-1]])
    
    # Distance de chaque point à la ligne
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)
    if line_len < 1e-10:
        return None
    
    line_unit = line_vec / line_len
    
    distances = []
    for i in range(len(x)):
        point = np.array([x_norm[i], y_norm[i]])
        vec = point - p1
        proj_len = np.dot(vec, line_unit)
        proj = p1 + proj_len * line_unit
        dist = np.linalg.norm(point - proj)
        distances.append(dist)
    
    idx = np.argmax(distances)
    return x[idx]


def find_percolation_threshold(x: np.ndarray, y: np.ndarray, 
                                threshold: float = 0.5) -> Optional[float]:
    """
    Trouve le cutoff où une fraction donnée d'analogues a au moins un contact.
    
    Parameters
    ----------
    x : np.ndarray
        Valeurs de cutoff
    y : np.ndarray
        Fraction d'isolés
    threshold : float
        Seuil de fraction en contact (défaut: 0.5 = 50%)
        
    Returns
    -------
    float or None
        Cutoff au seuil de percolation
    """
    # Fraction en contact = 1 - fraction isolés
    fraction_in_contact = 1 - y
    
    # Trouver le premier cutoff où fraction_in_contact >= threshold
    indices = np.where(fraction_in_contact >= threshold)[0]
    if len(indices) > 0:
        idx = indices[0]
        # Interpolation linéaire pour plus de précision
        if idx > 0:
            x1, x2 = x[idx-1], x[idx]
            y1, y2 = fraction_in_contact[idx-1], fraction_in_contact[idx]
            if y2 - y1 > 1e-10:
                return x1 + (threshold - y1) * (x2 - x1) / (y2 - y1)
        return x[idx]
    return None


print("✓ Fonctions d'analyse des analogues isolés définies")

# =============================================================================
# EXÉCUTION DE L'ANALYSE D'ISOLATION
# =============================================================================

# Plage de cutoffs pour l'analyse d'isolation
CUTOFF_RANGE_ISOLATION = np.arange(2.5, 10.1, 0.25)

print("\n" + "="*80)
print("ANALYSE D'ISOLATION EN FONCTION DU CUTOFF")
print("="*80)
print(f"\nPlage de cutoffs explorée : {CUTOFF_RANGE_ISOLATION.min():.2f} - {CUTOFF_RANGE_ISOLATION.max():.2f} Å")
print(f"Pas : {CUTOFF_RANGE_ISOLATION[1] - CUTOFF_RANGE_ISOLATION[0]:.2f} Å")
print(f"Nombre de systèmes à analyser : {len(all_frames)}")
print()

# Dictionnaire pour stocker les résultats
isolation_results = {}

# Analyse pour chaque système
for system_name, frames in tqdm(all_frames.items(), desc="Analyse d'isolation"):
    if len(frames) == 0:
        print(f"  ⚠ {system_name}: Aucune frame disponible")
        continue
    
    result = analyze_isolation_vs_cutoff(
        frames=frames,
        cutoffs=CUTOFF_RANGE_ISOLATION,
        distance_type='min_heavy',
        n_bootstrap=500,  # Réduit pour accélérer
        confidence_level=0.95
    )
    
    isolation_results[system_name] = result

print(f"\n✓ Analyse terminée pour {len(isolation_results)} systèmes")

# Résumé des cutoffs optimaux trouvés
print("\n" + "-"*60)
print("RÉSUMÉ DES CUTOFFS OPTIMAUX PAR SYSTÈME")
print("-"*60)
print(f"{'Système':<10} {'Inflexion (Å)':<15} {'Coude (Å)':<12} {'Percolation (Å)':<15}")
print("-"*60)

for system_name, result in sorted(isolation_results.items()):
    inflection_str = f"{result.optimal_cutoff_inflection:.2f}" if result.optimal_cutoff_inflection else "N/A"
    elbow_str = f"{result.optimal_cutoff_elbow:.2f}" if result.optimal_cutoff_elbow else "N/A"
    percolation_str = f"{result.optimal_cutoff_percolation:.2f}" if result.optimal_cutoff_percolation else "N/A"
    print(f"{system_name:<10} {inflection_str:<15} {elbow_str:<12} {percolation_str:<15}")

✓ Fonctions d'analyse des analogues isolés définies

ANALYSE D'ISOLATION EN FONCTION DU CUTOFF

Plage de cutoffs explorée : 2.50 - 10.00 Å
Pas : 0.25 Å
Nombre de systèmes à analyser : 27



Analyse d'isolation:   0%|          | 0/27 [00:00<?, ?it/s]


✓ Analyse terminée pour 27 systèmes

------------------------------------------------------------
RÉSUMÉ DES CUTOFFS OPTIMAUX PAR SYSTÈME
------------------------------------------------------------
Système    Inflexion (Å)   Coude (Å)    Percolation (Å)
------------------------------------------------------------
sca        3.50            7.50         N/A            
scc        3.50            7.50         N/A            
sccm       9.00            6.25         N/A            
scd        8.75            6.25         N/A            
scdn       3.00            6.00         N/A            
sce        3.75            6.50         N/A            
scen       3.25            5.00         N/A            
scf        3.50            3.50         7.67           
schd       3.50            6.25         N/A            
sche       3.50            6.25         N/A            
schp       7.25            6.00         N/A            
sci        3.75            3.75         8.36           
sck        

In [ ]:
# =============================================================================
# Graphique : Pourcentage de résidus en fonction du cutoff par type d'interaction
# =============================================================================

def compute_interaction_fractions_vs_cutoff(frames: List[Frame], 
                                            cutoffs: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Calcule la fraction de résidus ayant chaque type d'interaction pour une gamme de cutoffs.
    
    Parameters
    ----------
    frames : List[Frame]
        Liste des frames à analyser
    cutoffs : np.ndarray
        Plage de cutoffs à explorer (Å)
        
    Returns
    -------
    Dict[str, np.ndarray]
        Dictionnaire {type_interaction: array_fractions} avec:
        - 'isolated': fraction d'analogues isolés (aucun contact min_heavy)
        - 'vdw': fraction avec au moins un contact VdW
        - 'hydrophobic': fraction avec au moins un contact hydrophobe
        - 'hbond': fraction avec au moins une liaison H potentielle
        - 'pipi': fraction avec au moins un contact π-π (pour aromatiques)
        - 'any_contact': fraction avec au moins un contact quelconque
    """
    n_cutoffs = len(cutoffs)
    n_frames = len(frames)
    
    # Matrices pour chaque type d'interaction (frames x cutoffs)
    isolated_matrix = np.zeros((n_frames, n_cutoffs))
    vdw_matrix = np.zeros((n_frames, n_cutoffs))
    hydrophobic_matrix = np.zeros((n_frames, n_cutoffs))
    hbond_matrix = np.zeros((n_frames, n_cutoffs))
    pipi_matrix = np.zeros((n_frames, n_cutoffs))
    any_contact_matrix = np.zeros((n_frames, n_cutoffs))
    
    for frame_idx, frame in enumerate(frames):
        n_analogues = frame.n_analogues
        if n_analogues < 2:
            # Tous isolés si moins de 2 analogues
            isolated_matrix[frame_idx, :] = 1.0
            continue
        
        analogues = frame.analogues
        
        for cutoff_idx, cutoff in enumerate(cutoffs):
            # Tableaux pour tracker les contacts par analogue
            has_any_contact = np.zeros(n_analogues, dtype=bool)
            has_vdw = np.zeros(n_analogues, dtype=bool)
            has_hydrophobic = np.zeros(n_analogues, dtype=bool)
            has_hbond = np.zeros(n_analogues, dtype=bool)
            has_pipi = np.zeros(n_analogues, dtype=bool)
            
            # Analyser toutes les paires
            for i in range(n_analogues):
                for j in range(i + 1, n_analogues):
                    ana1, ana2 = analogues[i], analogues[j]
                    
                    # Contact VdW (distance min atomes lourds)
                    vdw_contact, vdw_dist, _ = detect_vdw_contacts(ana1, ana2, cutoff)
                    if vdw_contact:
                        has_vdw[i] = has_vdw[j] = True
                        has_any_contact[i] = has_any_contact[j] = True
                    
                    # Contact hydrophobe (cutoff adapté: typiquement +0.5 Å par rapport à VdW)
                    hydro_cutoff = cutoff + 0.5  # Contacts hydrophobes légèrement plus distants
                    hydro_contact, _, _ = detect_hydrophobic_contacts(ana1, ana2, hydro_cutoff)
                    if hydro_contact:
                        has_hydrophobic[i] = has_hydrophobic[j] = True
                        has_any_contact[i] = has_any_contact[j] = True
                    
                    # Liaison H (cutoff fixe plus court: ~3.5 Å typiquement)
                    hbond_cutoff = min(cutoff, 4.0)  # H-bonds rarement au-delà de 4 Å
                    hbond_contact, _, _ = detect_hbond_contacts(ana1, ana2, hbond_cutoff)
                    if hbond_contact:
                        has_hbond[i] = has_hbond[j] = True
                        has_any_contact[i] = has_any_contact[j] = True
                    
                    # Contact π-π (seulement pour aromatiques)
                    if ana1.has_aromatic_ring and ana2.has_aromatic_ring:
                        pipi_cutoff = cutoff + 1.0  # π-π typiquement jusqu'à 5-6 Å
                        pipi_contact, pipi_dist, _ = detect_pipi_contacts(ana1, ana2, pipi_cutoff)
                        if pipi_contact:
                            has_pipi[i] = has_pipi[j] = True
                            has_any_contact[i] = has_any_contact[j] = True
            
            # Calculer les fractions pour cette frame et ce cutoff
            isolated_matrix[frame_idx, cutoff_idx] = np.sum(~has_any_contact) / n_analogues
            vdw_matrix[frame_idx, cutoff_idx] = np.sum(has_vdw) / n_analogues
            hydrophobic_matrix[frame_idx, cutoff_idx] = np.sum(has_hydrophobic) / n_analogues
            hbond_matrix[frame_idx, cutoff_idx] = np.sum(has_hbond) / n_analogues
            pipi_matrix[frame_idx, cutoff_idx] = np.sum(has_pipi) / n_analogues
            any_contact_matrix[frame_idx, cutoff_idx] = np.sum(has_any_contact) / n_analogues
    
    # Moyennes sur toutes les frames
    return {
        'isolated': np.mean(isolated_matrix, axis=0),
        'isolated_std': np.std(isolated_matrix, axis=0, ddof=1),
        'vdw': np.mean(vdw_matrix, axis=0),
        'vdw_std': np.std(vdw_matrix, axis=0, ddof=1),
        'hydrophobic': np.mean(hydrophobic_matrix, axis=0),
        'hydrophobic_std': np.std(hydrophobic_matrix, axis=0, ddof=1),
        'hbond': np.mean(hbond_matrix, axis=0),
        'hbond_std': np.std(hbond_matrix, axis=0, ddof=1),
        'pipi': np.mean(pipi_matrix, axis=0),
        'pipi_std': np.std(pipi_matrix, axis=0, ddof=1),
        'any_contact': np.mean(any_contact_matrix, axis=0),
        'any_contact_std': np.std(any_contact_matrix, axis=0, ddof=1),
    }


# =============================================================================
# Calcul des fractions pour tous les systèmes combinés
# =============================================================================

print("\n" + "="*80)
print("CALCUL DES FRACTIONS D'INTERACTION PAR TYPE EN FONCTION DU CUTOFF")
print("="*80)

# Utiliser la même plage de cutoffs que pour l'analyse d'isolation
CUTOFFS_PLOT = np.arange(2.5, 8.1, 0.25)

# Combiner toutes les frames de tous les systèmes pour une vue globale
all_frames_combined = []
for system_name, frames in all_frames.items():
    all_frames_combined.extend(frames)

print(f"\nNombre total de frames combinées: {len(all_frames_combined)}")
print(f"Plage de cutoffs: {CUTOFFS_PLOT.min():.2f} - {CUTOFFS_PLOT.max():.2f} Å")
print("\nCalcul en cours...")

# Calcul des fractions (cela peut prendre un moment)
interaction_fractions = compute_interaction_fractions_vs_cutoff(all_frames_combined, CUTOFFS_PLOT)

print("✓ Calcul terminé")

# =============================================================================
# Création du graphique
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 8))

# Couleurs pour chaque type d'interaction
colors = {
    'isolated': '#2ecc71',      # Vert - isolés
    'vdw': '#3498db',           # Bleu - VdW
    'hydrophobic': '#e67e22',   # Orange - hydrophobe
    'hbond': '#9b59b6',         # Violet - H-bond
    'pipi': '#e74c3c',          # Rouge - π-π
    'any_contact': '#34495e',   # Gris foncé - tout contact
}

labels = {
    'isolated': 'Isolés (aucun contact)',
    'vdw': 'Contact VdW',
    'hydrophobic': 'Contact hydrophobe',
    'hbond': 'Liaison H potentielle',
    'pipi': 'Interaction π-π',
    'any_contact': 'Tout contact',
}

# Tracer les courbes avec intervalles de confiance
for interaction_type in ['isolated', 'vdw', 'hydrophobic', 'hbond', 'pipi']:
    mean = interaction_fractions[interaction_type] * 100  # En pourcentage
    std = interaction_fractions[f'{interaction_type}_std'] * 100
    
    # Courbe principale
    ax.plot(CUTOFFS_PLOT, mean, 
            color=colors[interaction_type], 
            linewidth=2.5, 
            label=labels[interaction_type])
    
    # Bande d'incertitude (±1 écart-type)
    ax.fill_between(CUTOFFS_PLOT, mean - std, mean + std, 
                    color=colors[interaction_type], 
                    alpha=0.15)

# Configuration des axes
ax.set_xlabel('Cutoff de distance (Å)', fontsize=14, fontweight='bold')
ax.set_ylabel('Pourcentage de résidus (%)', fontsize=14, fontweight='bold')
ax.set_title('Fraction de résidus par type d\'interaction en fonction du cutoff\n(Moyenne sur tous les systèmes)', 
             fontsize=16, fontweight='bold')

# Limites et grille
ax.set_xlim(CUTOFFS_PLOT.min(), CUTOFFS_PLOT.max())
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, linestyle='--')

# Légende
ax.legend(loc='center right', fontsize=11, framealpha=0.95)

# Annotations des cutoffs typiques
typical_cutoffs = {
    3.5: 'H-bond\ntypique',
    4.0: 'VdW\ntypique', 
    4.5: 'Hydrophobe\ntypique',
    5.0: 'π-π\ntypique',
}

for cutoff, label in typical_cutoffs.items():
    ax.axvline(x=cutoff, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax.annotate(label, xy=(cutoff, 100), xytext=(cutoff, 102),
                fontsize=8, ha='center', va='bottom', color='gray')

plt.tight_layout()

# Sauvegarde
output_path = FIGURES_DIR / "interaction_types_vs_cutoff.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
output_path_pdf = FIGURES_DIR / "interaction_types_vs_cutoff.pdf"
plt.savefig(output_path_pdf, dpi=300, bbox_inches='tight', facecolor='white')

print(f"\n✓ Figure sauvegardée: {output_path}")
print(f"✓ Figure sauvegardée: {output_path_pdf}")

plt.show()

# =============================================================================
# Statistiques récapitulatives
# =============================================================================

print("\n" + "="*80)
print("STATISTIQUES À DIFFÉRENTS CUTOFFS")
print("="*80)

reference_cutoffs = [3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
print(f"\n{'Cutoff (Å)':<12} {'Isolés (%)':<15} {'VdW (%)':<15} {'Hydrophobe (%)':<18} {'H-bond (%)':<15} {'π-π (%)':<12}")
print("-"*90)

for cutoff in reference_cutoffs:
    idx = np.argmin(np.abs(CUTOFFS_PLOT - cutoff))
    isolated = interaction_fractions['isolated'][idx] * 100
    vdw = interaction_fractions['vdw'][idx] * 100
    hydro = interaction_fractions['hydrophobic'][idx] * 100
    hbond = interaction_fractions['hbond'][idx] * 100
    pipi = interaction_fractions['pipi'][idx] * 100
    
    print(f"{cutoff:<12.1f} {isolated:<15.1f} {vdw:<15.1f} {hydro:<18.1f} {hbond:<15.1f} {pipi:<12.1f}")

In [ ]:
# =============================================================================
# Graphique par catégorie d'acide aminé
# =============================================================================

print("\n" + "="*80)
print("ANALYSE PAR CATÉGORIE D'ACIDE AMINÉ")
print("="*80)

# Grouper les systèmes par catégorie
categories = {
    'aromatic': [],
    'aliphatic': [],
    'charged': [],
    'polar': [],
}

for system_name, frames in all_frames.items():
    if system_name in ANALOGUE_INFO:
        category = ANALOGUE_INFO[system_name][2]
        if category in categories:
            categories[category].extend(frames)

# Calculer les fractions par catégorie
category_fractions = {}
for category, frames in categories.items():
    if len(frames) > 0:
        print(f"\n{category.capitalize()}: {len(frames)} frames")
        category_fractions[category] = compute_interaction_fractions_vs_cutoff(frames, CUTOFFS_PLOT)
    else:
        print(f"\n{category.capitalize()}: Aucune frame")

# =============================================================================
# Figure avec 4 sous-graphiques (un par catégorie)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

category_names = {
    'aromatic': 'Aromatiques (Phe, Tyr, Trp, His)',
    'aliphatic': 'Aliphatiques (Ala, Val, Leu, Ile, Met, Pro)',
    'charged': 'Chargés (Asp, Glu, Lys, Arg, His+)',
    'polar': 'Polaires (Ser, Thr, Asn, Gln, Cys)',
}

category_order = ['aromatic', 'aliphatic', 'charged', 'polar']

for ax_idx, category in enumerate(category_order):
    ax = axes[ax_idx]
    
    if category not in category_fractions:
        ax.text(0.5, 0.5, 'Données non disponibles', 
                ha='center', va='center', fontsize=14, transform=ax.transAxes)
        ax.set_title(category_names.get(category, category.capitalize()), fontsize=14, fontweight='bold')
        continue
    
    fractions = category_fractions[category]
    
    # Tracer les courbes
    for interaction_type in ['isolated', 'vdw', 'hydrophobic', 'hbond', 'pipi']:
        # Ne pas tracer π-π pour les non-aromatiques
        if interaction_type == 'pipi' and category != 'aromatic':
            continue
            
        mean = fractions[interaction_type] * 100
        std = fractions[f'{interaction_type}_std'] * 100
        
        ax.plot(CUTOFFS_PLOT, mean, 
                color=colors[interaction_type], 
                linewidth=2, 
                label=labels[interaction_type])
        ax.fill_between(CUTOFFS_PLOT, mean - std, mean + std, 
                        color=colors[interaction_type], 
                        alpha=0.15)
    
    # Configuration
    ax.set_xlabel('Cutoff (Å)', fontsize=12)
    ax.set_ylabel('Pourcentage (%)', fontsize=12)
    ax.set_title(category_names.get(category, category.capitalize()), fontsize=14, fontweight='bold')
    ax.set_xlim(CUTOFFS_PLOT.min(), CUTOFFS_PLOT.max())
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='center right', fontsize=9, framealpha=0.9)

plt.suptitle('Fraction de résidus par type d\'interaction selon la catégorie d\'acide aminé', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Sauvegarde
output_path = FIGURES_DIR / "interaction_types_by_category.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
output_path_pdf = FIGURES_DIR / "interaction_types_by_category.pdf"
plt.savefig(output_path_pdf, dpi=300, bbox_inches='tight', facecolor='white')

print(f"\n✓ Figure sauvegardée: {output_path}")
print(f"✓ Figure sauvegardée: {output_path_pdf}")

plt.show()

# =============================================================================
# Graphique comparatif: isolés par catégorie
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 7))

category_colors = {
    'aromatic': '#e74c3c',    # Rouge
    'aliphatic': '#3498db',   # Bleu
    'charged': '#9b59b6',     # Violet
    'polar': '#2ecc71',       # Vert
}

for category in category_order:
    if category not in category_fractions:
        continue
    
    fractions = category_fractions[category]
    mean = fractions['isolated'] * 100
    std = fractions['isolated_std'] * 100
    
    ax.plot(CUTOFFS_PLOT, mean, 
            color=category_colors[category], 
            linewidth=2.5, 
            label=category_names.get(category, category.capitalize()))
    ax.fill_between(CUTOFFS_PLOT, mean - std, mean + std, 
                    color=category_colors[category], 
                    alpha=0.15)

ax.set_xlabel('Cutoff de distance (Å)', fontsize=14, fontweight='bold')
ax.set_ylabel('Pourcentage de résidus isolés (%)', fontsize=14, fontweight='bold')
ax.set_title('Fraction de résidus isolés par catégorie d\'acide aminé', 
             fontsize=16, fontweight='bold')
ax.set_xlim(CUTOFFS_PLOT.min(), CUTOFFS_PLOT.max())
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=11, framealpha=0.95)

# Ligne de référence à 50%
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.annotate('50%', xy=(CUTOFFS_PLOT.max(), 50), xytext=(CUTOFFS_PLOT.max()+0.1, 50),
            fontsize=10, ha='left', va='center', color='gray')

plt.tight_layout()

# Sauvegarde
output_path = FIGURES_DIR / "isolated_fraction_by_category.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
output_path_pdf = FIGURES_DIR / "isolated_fraction_by_category.pdf"
plt.savefig(output_path_pdf, dpi=300, bbox_inches='tight', facecolor='white')

print(f"\n✓ Figure sauvegardée: {output_path}")
print(f"✓ Figure sauvegardée: {output_path_pdf}")

plt.show()

In [14]:
# =============================================================================
# Synthèse et recommandations finales
# =============================================================================

def generate_cutoff_recommendations(results: Dict[str, IsolationAnalysisResult]) -> Dict:
    """
    Génère des recommandations basées sur l'analyse d'isolation.
    
    Parameters
    ----------
    results : Dict[str, IsolationAnalysisResult]
        Résultats d'analyse par système
        
    Returns
    -------
    Dict
        Recommandations détaillées
    """
    # Vérification que les résultats ne sont pas vides
    if not results:
        print("⚠ ATTENTION: Aucun résultat d'isolation disponible!")
        print("  Veuillez exécuter d'abord la cellule d'analyse d'isolation.")
        return {
            'cutoff_inflection': None,
            'cutoff_elbow': None,
            'cutoff_percolation': None,
            'isolation_at_cutoffs': {},
            'recommended_cutoff': 4.0,
            'conservative_cutoff': 3.5,
            'permissive_cutoff': 5.0,
        }
    
    # Collecter tous les cutoffs optimaux (filtrer les None)
    all_inflection = [r.optimal_cutoff_inflection for r in results.values() 
                      if r.optimal_cutoff_inflection is not None]
    all_elbow = [r.optimal_cutoff_elbow for r in results.values() 
                 if r.optimal_cutoff_elbow is not None]
    all_percolation = [r.optimal_cutoff_percolation for r in results.values() 
                       if r.optimal_cutoff_percolation is not None]
    
    # Calculer les fractions isolées à différents cutoffs pour tous les systèmes
    cutoff_reference = [3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
    isolation_at_cutoffs = {c: [] for c in cutoff_reference}
    
    for result in results.values():
        for cutoff in cutoff_reference:
            idx = np.argmin(np.abs(result.cutoffs - cutoff))
            isolation_at_cutoffs[cutoff].append(result.fraction_isolated[idx])
    
    # Recommandations (avec gestion des listes vides)
    recommendations = {}
    
    if all_inflection:
        recommendations['cutoff_inflection'] = {
            'mean': np.mean(all_inflection),
            'std': np.std(all_inflection),
            'median': np.median(all_inflection),
            'iqr': (np.percentile(all_inflection, 25), np.percentile(all_inflection, 75)),
        }
    else:
        recommendations['cutoff_inflection'] = None
    
    if all_elbow:
        recommendations['cutoff_elbow'] = {
            'mean': np.mean(all_elbow),
            'std': np.std(all_elbow),
            'median': np.median(all_elbow),
            'iqr': (np.percentile(all_elbow, 25), np.percentile(all_elbow, 75)),
        }
    else:
        recommendations['cutoff_elbow'] = None
    
    if all_percolation:
        recommendations['cutoff_percolation'] = {
            'mean': np.mean(all_percolation),
            'std': np.std(all_percolation),
            'median': np.median(all_percolation),
            'iqr': (np.percentile(all_percolation, 25), np.percentile(all_percolation, 75)),
        }
    else:
        recommendations['cutoff_percolation'] = None
    
    # Gestion robuste des statistiques d'isolation (éviter NaN avec listes vides)
    recommendations['isolation_at_cutoffs'] = {}
    for c, v in isolation_at_cutoffs.items():
        if len(v) > 0:
            recommendations['isolation_at_cutoffs'][c] = {
                'mean': np.mean(v),
                'std': np.std(v) if len(v) > 1 else 0.0
            }
        else:
            recommendations['isolation_at_cutoffs'][c] = {'mean': np.nan, 'std': np.nan}
    
    # Cutoff recommandé basé sur la méthode du coude si disponible
    if all_elbow:
        recommendations['recommended_cutoff'] = np.median(all_elbow)
        recommendations['conservative_cutoff'] = np.percentile(all_elbow, 25)
        recommendations['permissive_cutoff'] = np.percentile(all_elbow, 75)
    elif all_inflection:
        # Fallback sur inflexion si coude non disponible
        recommendations['recommended_cutoff'] = np.median(all_inflection)
        recommendations['conservative_cutoff'] = np.percentile(all_inflection, 25)
        recommendations['permissive_cutoff'] = np.percentile(all_inflection, 75)
    else:
        # Fallback sur valeur par défaut
        recommendations['recommended_cutoff'] = 4.0
        recommendations['conservative_cutoff'] = 3.5
        recommendations['permissive_cutoff'] = 5.0
    
    return recommendations


# Vérification préalable des données
if not isolation_results:
    print("="*80)
    print("⚠ ERREUR: Le dictionnaire 'isolation_results' est vide!")
    print("="*80)
    print("\nCela signifie que l'analyse d'isolation n'a pas été exécutée.")
    print("Veuillez exécuter la cellule précédente qui contient l'analyse d'isolation.")
    print("\nSi vous avez déjà exécuté cette cellule, vérifiez que 'all_frames' contient des données.")
else:
    print(f"✓ {len(isolation_results)} systèmes analysés disponibles")

# Génération des recommandations
recommendations = generate_cutoff_recommendations(isolation_results)

# Affichage formaté
print("\n" + "=" * 80)
print("SYNTHÈSE ET RECOMMANDATIONS POUR LE CUTOFF DE CONTACT")
print("=" * 80)

print("\n┌─────────────────────────────────────────────────────────────────────────────┐")
print("│                    CUTOFFS OPTIMAUX PAR MÉTHODE                             │")
print("├─────────────────────────────────────────────────────────────────────────────┤")

if recommendations['cutoff_inflection']:
    inf = recommendations['cutoff_inflection']
    print(f"│  Point d'inflexion:  {inf['median']:.2f} Å "
          f"(IQR: {inf['iqr'][0]:.2f}-{inf['iqr'][1]:.2f} Å)                    │")
else:
    print("│  Point d'inflexion:  N/A                                                    │")

if recommendations['cutoff_elbow']:
    elb = recommendations['cutoff_elbow']
    print(f"│  Méthode du coude:   {elb['median']:.2f} Å "
          f"(IQR: {elb['iqr'][0]:.2f}-{elb['iqr'][1]:.2f} Å)                    │")
else:
    print("│  Méthode du coude:   N/A                                                    │")

if recommendations['cutoff_percolation']:
    per = recommendations['cutoff_percolation']
    print(f"│  Percolation 50%:    {per['median']:.2f} Å "
          f"(IQR: {per['iqr'][0]:.2f}-{per['iqr'][1]:.2f} Å)                    │")
else:
    print("│  Percolation 50%:    N/A                                                    │")

print("└─────────────────────────────────────────────────────────────────────────────┘")

# Affichage de la fraction d'isolés seulement si des données existent
if recommendations['isolation_at_cutoffs']:
    print("\n┌─────────────────────────────────────────────────────────────────────────────┐")
    print("│              FRACTION D'ANALOGUES ISOLÉS À DIFFÉRENTS CUTOFFS              │")
    print("├─────────────────────────────────────────────────────────────────────────────┤")
    for cutoff, stats in recommendations['isolation_at_cutoffs'].items():
        if not np.isnan(stats['mean']):
            bar_len = int(stats['mean'] * 40)
            bar = '█' * bar_len + '░' * (40 - bar_len)
            print(f"│  {cutoff:.1f} Å: {bar} {100*stats['mean']:.1f} ± {100*stats['std']:.1f}% │")
        else:
            print(f"│  {cutoff:.1f} Å: {'N/A':<47} │")
    print("└─────────────────────────────────────────────────────────────────────────────┘")

print("\n┌─────────────────────────────────────────────────────────────────────────────┐")
print("│                         RECOMMANDATIONS FINALES                             │")
print("├─────────────────────────────────────────────────────────────────────────────┤")
print(f"│  ★ Cutoff recommandé (contacts significatifs):     {recommendations['recommended_cutoff']:.2f} Å                │")
print(f"│  → Cutoff conservateur (contacts forts):           {recommendations['conservative_cutoff']:.2f} Å                │")
print(f"│  → Cutoff permissif (tous contacts potentiels):    {recommendations['permissive_cutoff']:.2f} Å                │")
print("└─────────────────────────────────────────────────────────────────────────────┘")

print("\n" + "─" * 80)
print("INTERPRÉTATION:")
print("─" * 80)
print("""
• Le cutoff RECOMMANDÉ correspond au point où la transition entre analogues
  isolés et analogues en contact est la plus marquée (méthode du coude).
  
• Le cutoff CONSERVATEUR détecte uniquement les contacts les plus forts,
  correspondant à des interactions de van der Waals significatives.
  
• Le cutoff PERMISSIF inclut les contacts plus distants, potentiellement
  des interactions faibles ou des proximités géométriques sans interaction.
  
• Pour les calculs d'énergie libre, le cutoff conservateur est recommandé
  pour éviter de surestimer les effets d'interactions inter-analogues.
""")

# Sauvegarde des résultats uniquement si des données existent
if isolation_results:
    output_file = OUTPUT_DIR / "isolation_analysis_results.json"
    with open(output_file, 'w') as f:
        # Convertir les np.ndarray en listes pour JSON
        json_safe = {}
        for k, v in recommendations.items():
            if v is None:
                json_safe[k] = None
            elif isinstance(v, dict):
                json_safe[k] = {str(kk): (list(vv) if isinstance(vv, (np.ndarray, tuple)) else vv) 
                               for kk, vv in v.items()}
            else:
                json_safe[k] = float(v) if isinstance(v, (np.floating, float)) else v
        json.dump(json_safe, f, indent=2)
    print(f"\n✓ Résultats sauvegardés: {output_file}")

✓ 27 systèmes analysés disponibles

SYNTHÈSE ET RECOMMANDATIONS POUR LE CUTOFF DE CONTACT

┌─────────────────────────────────────────────────────────────────────────────┐
│                    CUTOFFS OPTIMAUX PAR MÉTHODE                             │
├─────────────────────────────────────────────────────────────────────────────┤
│  Point d'inflexion:  3.50 Å (IQR: 3.38-6.25 Å)                    │
│  Méthode du coude:   6.00 Å (IQR: 4.00-6.25 Å)                    │
│  Percolation 50%:    8.19 Å (IQR: 7.34-8.68 Å)                    │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│              FRACTION D'ANALOGUES ISOLÉS À DIFFÉRENTS CUTOFFS              │
├─────────────────────────────────────────────────────────────────────────────┤
│  3.5 Å: ███████████████████████████████████████░ 98.1 ± 3.4% │
│  4.0 Å: █████████████████████████████████████░░░ 93.0 ± 9.7% │
│  4.5 Å:

### 8.7 Validation statistique : Test de sensibilité au cutoff

Pour valider la robustesse du cutoff recommandé, nous effectuons une analyse de sensibilité en examinant comment les métriques clés varient dans un voisinage du cutoff optimal.

In [15]:
# =============================================================================
# Analyse de sensibilité au cutoff
# =============================================================================

def sensitivity_analysis(results: Dict[str, IsolationAnalysisResult],
                         center_cutoff: float,
                         window: float = 1.0) -> pd.DataFrame:
    """
    Analyse la sensibilité des métriques autour d'un cutoff donné.
    
    Parameters
    ----------
    results : Dict[str, IsolationAnalysisResult]
        Résultats d'isolation par système
    center_cutoff : float
        Cutoff central pour l'analyse
    window : float
        Demi-largeur de la fenêtre d'analyse (Å)
        
    Returns
    -------
    pd.DataFrame
        Métriques de sensibilité par système
    """
    # Vérification des données d'entrée
    if not results:
        print("⚠ Aucun résultat d'isolation disponible pour l'analyse de sensibilité")
        return pd.DataFrame()
    
    data = []
    
    for system_name, result in sorted(results.items()):
        cutoffs = result.cutoffs
        fractions = result.fraction_isolated
        
        # Indices dans la fenêtre
        mask = (cutoffs >= center_cutoff - window) & (cutoffs <= center_cutoff + window)
        window_cutoffs = cutoffs[mask]
        window_fractions = fractions[mask]
        
        if len(window_cutoffs) < 3:
            continue
        
        # Dérivée locale (sensibilité)
        derivative = np.gradient(window_fractions, window_cutoffs)
        
        # Coefficient de variation dans la fenêtre
        mean_frac = np.mean(window_fractions)
        cv = np.std(window_fractions) / mean_frac if mean_frac > 0 else 0
        
        # Valeur au centre
        idx_center = np.argmin(np.abs(cutoffs - center_cutoff))
        fraction_center = fractions[idx_center]
        
        info = ANALOGUE_INFO.get(system_name, (system_name.upper(), 'Unknown', 'unknown'))
        
        data.append({
            'Système': system_name,
            'Type': info[2],
            f'Isolés @ {center_cutoff:.1f}Å (%)': f"{100*fraction_center:.1f}",
            'Sensibilité moyenne (%/Å)': f"{100*np.mean(np.abs(derivative)):.2f}",
            'Sensibilité max (%/Å)': f"{100*np.max(np.abs(derivative)):.2f}",
            'CV dans fenêtre (%)': f"{100*cv:.1f}",
        })
    
    return pd.DataFrame(data)


# Vérification préalable des données
if not isolation_results:
    print("="*80)
    print("⚠ ERREUR: Le dictionnaire 'isolation_results' est vide!")
    print("="*80)
    print("\nL'analyse de sensibilité ne peut pas être effectuée.")
    print("Veuillez exécuter les cellules précédentes d'abord.")
else:
    # Analyse de sensibilité autour du cutoff recommandé
    center = recommendations['recommended_cutoff']
    sensitivity_df = sensitivity_analysis(isolation_results, center, window=0.5)

    print(f"\n{'='*80}")
    print(f"ANALYSE DE SENSIBILITÉ AUTOUR DU CUTOFF {center:.2f} Å (± 0.5 Å)")
    print(f"{'='*80}\n")

    if not sensitivity_df.empty:
        display(sensitivity_df.style.set_caption(f"Sensibilité des métriques d'isolation autour de {center:.2f} Å"))

        # Calcul de la sensibilité moyenne globale
        # Extraction robuste des valeurs numériques
        try:
            sensitivities = [float(s.replace('%/Å', '')) 
                           for s in sensitivity_df['Sensibilité moyenne (%/Å)']
                           if s and s != 'N/A']
            
            if sensitivities:
                print(f"\nSensibilité moyenne globale: {np.mean(sensitivities):.2f} %/Å")
                print(f"  → Une variation de ±0.25 Å autour du cutoff recommandé")
                print(f"     entraîne une variation moyenne de ±{0.25*np.mean(sensitivities):.1f}% dans la fraction d'isolés")
            else:
                print("\n⚠ Impossible de calculer la sensibilité moyenne (données insuffisantes)")
        except (KeyError, ValueError) as e:
            print(f"\n⚠ Erreur lors du calcul de la sensibilité: {e}")
            print("  Vérifiez que les colonnes du DataFrame sont correctes.")
    else:
        print("⚠ Le DataFrame de sensibilité est vide.")
        print("  Cela peut indiquer que les cutoffs ne sont pas dans la plage analysée.")
        print(f"  Cutoff central: {center:.2f} Å")
        print(f"  Fenêtre: ± 0.5 Å")
        if isolation_results:
            first_result = next(iter(isolation_results.values()))
            print(f"  Plage de cutoffs disponible: {first_result.cutoffs.min():.2f} - {first_result.cutoffs.max():.2f} Å")


ANALYSE DE SENSIBILITÉ AUTOUR DU CUTOFF 6.00 Å (± 0.5 Å)



,Système,Type,Isolés @ 6.0Å (%),Sensibilité moyenne (%/Å),Sensibilité max (%/Å),CV dans fenêtre (%)
0,sca,aliphatic,93.4,2.09,2.92,0.8
1,scc,polar,85.3,3.48,4.00,1.4
2,sccm,charged,98.2,1.28,2.62,0.4
3,scd,charged,95.5,2.77,4.00,1.0
4,scdn,polar,90.5,3.15,4.00,1.2
5,sce,charged,95.8,2.35,2.54,0.9
6,scen,polar,79.0,3.83,4.62,1.7
7,scf,aromatic,64.4,5.95,7.08,3.1
8,schd,aromatic,92.0,2.95,4.31,1.1
9,sche,aromatic,92.3,3.14,4.00,1.2



Sensibilité moyenne globale: 3.69 %/Å
  → Une variation de ±0.25 Å autour du cutoff recommandé
     entraîne une variation moyenne de ±0.9% dans la fraction d'isolés
